# LeetCode 280 — Wiggle Sort bằng Python

Link đề: https://leetcode.com/problems/wiggle-sort/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def wiggleSort(self, nums: List[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
```

Ghi chú: đây là bài **Premium** của LeetCode; đề bài viết theo hiểu biết về bài (và bản mô tả đã tải), nên đối chiếu lại nếu có tài khoản Premium. Vì đáp án **không duy nhất** (mọi hoán vị thỏa điều kiện đều được chấp nhận), phần test kiểm tra theo **tính chất** (đúng là hoán vị của đầu vào và thỏa `nums[0] <= nums[1] >= nums[2] <= ...`) và đối chiếu với một bộ kiểm độc lập duyệt mọi hoán vị cho mảng nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: sắp xếp lại mảng `nums` **tại chỗ (in-place)** sao cho `nums[0] <= nums[1] >= nums[2] <= nums[3] >= ...` (xen kẽ **lên – xuống**, hay "lắc"); đề đảm bảo luôn có đáp án; **thứ tự đáp án không duy nhất**.
2. Hiểu **điều kiện cục bộ**: chỉ cần mỗi **vị trí lẻ** `i` có `nums[i] >= nums[i − 1]` (và `>= nums[i + 1]`), mỗi **vị trí chẵn** `i` có `nums[i] <= nums[i − 1]`; **không cần sắp xếp toàn bộ**.
3. Đi từ **thử mọi hoán vị (brute force)** → **sắp xếp rồi đổi chỗ từng cặp** → **sắp xếp rồi xen kẽ nửa nhỏ và nửa lớn** → **một lượt duyệt, sửa cục bộ bằng hoán đổi (cách tham khảo, `O(n)`)** → **một lượt duyệt nhìn về phía trước**.
4. Hiểu **vì sao hoán đổi cục bộ không phá vỡ phần đã sắp đúng** (bất biến của vòng lặp), đây là lập luận tham lam.
5. Tránh các lỗi kinh điển: xét sai chẵn/lẻ, nhầm chiều so sánh (`<=` với `>=`), chỉ `sort()` rồi dừng, truy cập `nums[i + 1]` vượt biên, và **trả về mảng mới thay vì sửa tại chỗ**.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, hãy **sắp xếp lại** nó sao cho:

```
nums[0] <= nums[1] >= nums[2] <= nums[3] >= ...
```

Bạn có thể giả sử đầu vào luôn có đáp án hợp lệ. Hàm **không trả về gì**; hãy sửa `nums` **tại chỗ**.

**Ví dụ 1**

- Input: `nums = [3,5,2,1,6,4]`
- Output: `[3,5,1,6,2,4]`
- Giải thích: `[1,6,2,5,3,4]` cũng được chấp nhận (có nhiều đáp án hợp lệ).

**Ví dụ 2**

- Input: `nums = [6,6,5,6,3,8]`
- Output: `[6,6,5,6,3,8]`

**Ràng buộc:** `1 <= nums.length <= 5 * 10⁴`; `0 <= nums[i] <= 10⁴`; đảm bảo có đáp án cho `nums`.

**Follow up:** có thể giải trong thời gian `O(n)` không?

In [1]:
from typing import List
import random
from itertools import permutations

examples = [
    ([3, 5, 2, 1, 6, 4], [3, 5, 1, 6, 2, 4]),
    ([6, 6, 5, 6, 3, 8], [6, 6, 5, 6, 3, 8]),
]


def is_wiggle(a):
    """Kiểm tra a[0] <= a[1] >= a[2] <= a[3] >= ... (chỉ số chẵn: <= phần tử kế; chỉ số lẻ: >= phần tử kế)."""
    return all((a[i] <= a[i + 1]) if i % 2 == 0 else (a[i] >= a[i + 1]) for i in range(len(a) - 1))


for nums, expected in examples:
    print(f"nums={nums}, ví dụ đáp án {expected}, hợp lệ? {is_wiggle(expected)}")

nums=[3, 5, 2, 1, 6, 4], ví dụ đáp án [3, 5, 1, 6, 2, 4], hợp lệ? True
nums=[6, 6, 5, 6, 3, 8], ví dụ đáp án [6, 6, 5, 6, 3, 8], hợp lệ? True


## 3. Kiến thức nền

### 3.1. Điều kiện xen kẽ theo chẵn/lẻ

`nums[0] <= nums[1] >= nums[2] <= nums[3] >= ...` nghĩa là: với chỉ số **chẵn** `i` thì `nums[i] <= nums[i + 1]`; với chỉ số **lẻ** `i` thì `nums[i] >= nums[i + 1]`. Nói theo từng phần tử `i` (từ `1`): `i` **lẻ** thì `nums[i] >= nums[i − 1]` ("đỉnh" cao hơn phần tử trước), `i` **chẵn** thì `nums[i] <= nums[i − 1]` ("thung lũng" thấp hơn phần tử trước). Các vị trí lẻ là **đỉnh**, các vị trí chẵn là **đáy**.

In [2]:
for a in ([1, 3, 2, 5, 4, 6], [1, 2, 3], [3, 1, 2], [2, 2, 2], [5]):
    print(a, "->", is_wiggle(a))

[1, 3, 2, 5, 4, 6] -> True
[1, 2, 3] -> False
[3, 1, 2] -> False
[2, 2, 2] -> True
[5] -> True


### 3.2. Hoán đổi hai phần tử tại chỗ

`a[i], a[j] = a[j], a[i]` đổi chỗ hai phần tử mà không cần biến tạm (vế phải được tính xong trước khi gán). Hàm trong bài **sửa trực tiếp** `nums` (không `return` mảng mới): sau khi hàm kết thúc, người gọi thấy `nums` đã thay đổi.

In [3]:
a = [3, 5, 2]
a[1], a[2] = a[2], a[1]
print(a)                                    # [3, 2, 5]


def modify_in_place(nums):
    nums.sort()                             # sửa đối tượng list được truyền vào


def return_new(nums):
    return sorted(nums)                     # tạo list mới, nums gốc không đổi


x = [3, 1, 2]
modify_in_place(x)
print(x)                                    # [1, 2, 3]
y = [3, 1, 2]
return_new(y)
print(y)                                    # [3, 1, 2] (không đổi)

[3, 2, 5]
[1, 2, 3]
[3, 1, 2]


### 3.3. Sắp xếp và cắt lát `a[0::2]`, `a[1::2]`

`sorted(a)` (hoặc `a.sort()`) sắp xếp tăng dần. Cắt lát có bước: `a[0::2]` là các phần tử ở chỉ số chẵn, `a[1::2]` là ở chỉ số lẻ. Gán vào lát cắt có bước: `a[0::2] = [...]` ghi đè các vị trí chẵn (số phần tử phải đúng bằng số vị trí).

In [4]:
a = list(range(10, 17))
print(a[0::2], a[1::2])                     # [10, 12, 14, 16] [11, 13, 15]
b = [0] * 7
b[0::2] = [1, 2, 3, 4]                       # 4 vị trí chẵn
b[1::2] = [9, 8, 7]                          # 3 vị trí lẻ
print(b)                                     # [1, 9, 2, 8, 3, 7, 4]

[10, 12, 14, 16] [11, 13, 15]
[1, 9, 2, 8, 3, 7, 4]


### 3.4. Bất biến vòng lặp và lập luận tham lam cục bộ

Nếu trước bước `i` ta đã có `nums[0..i−1]` xen kẽ đúng, ta chỉ cần xử lý cặp `(nums[i − 1], nums[i])`: nếu vi phạm thì **đổi chỗ** hai phần tử. Mấu chốt (sẽ chứng minh ở mục 10): phép đổi này **không làm hỏng** cặp `(nums[i − 2], nums[i − 1])` đã đúng. Đây là kiểu lập luận **tham lam**: mỗi bước sửa cục bộ mà không cần nhìn cả mảng.

In [5]:
nums = [3, 5, 2, 1, 6, 4]
for i in range(1, len(nums)):
    bad = (i % 2 == 1 and nums[i] < nums[i - 1]) or (i % 2 == 0 and nums[i] > nums[i - 1])
    print(f"i={i}: cặp ({nums[i - 1]}, {nums[i]}) {'vi phạm -> đổi chỗ' if bad else 'đúng'}", end="")
    if bad:
        nums[i], nums[i - 1] = nums[i - 1], nums[i]
    print(" ->", nums)

i=1: cặp (3, 5) đúng -> [3, 5, 2, 1, 6, 4]
i=2: cặp (5, 2) đúng -> [3, 5, 2, 1, 6, 4]
i=3: cặp (2, 1) vi phạm -> đổi chỗ -> [3, 5, 1, 2, 6, 4]
i=4: cặp (2, 6) vi phạm -> đổi chỗ -> [3, 5, 1, 6, 2, 4]
i=5: cặp (2, 4) đúng -> [3, 5, 1, 6, 2, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ `[3, 5, 2, 1, 6, 4]`. Muốn: `đáy ≤ đỉnh ≥ đáy ≤ đỉnh ≥ ...`, tức vị trí `0, 2, 4` là đáy; `1, 3, 5` là đỉnh.

Quét từ trái sang phải, mỗi bước chỉ nhìn cặp `(nums[i − 1], nums[i])`:

| `i` | loại vị trí | cặp | cần | hành động |
|---|---|---|---|---|
| 1 | đỉnh | `(3, 5)` | `5 ≥ 3` | đúng |
| 2 | đáy | `(5, 2)` | `2 ≤ 5` | đúng |
| 3 | đỉnh | `(2, 1)` | `1 ≥ 2` | **sai** → đổi: `[3,5,1,2,6,4]` |
| 4 | đáy | `(2, 6)` | `6 ≤ 2` | **sai** → đổi: `[3,5,1,6,2,4]` |
| 5 | đỉnh | `(2, 4)` | `4 ≥ 2` | đúng |

Kết quả `[3, 5, 1, 6, 2, 4]`: `3 ≤ 5 ≥ 1 ≤ 6 ≥ 2 ≤ 4`. Không cần sắp xếp cả mảng: chỉ `O(n)` phép so sánh và nhiều nhất `n − 1` hoán đổi.

In [6]:
nums = [3, 5, 2, 1, 6, 4]
for i in range(1, len(nums)):
    if (i % 2 == 1 and nums[i] < nums[i - 1]) or (i % 2 == 0 and nums[i] > nums[i - 1]):
        nums[i], nums[i - 1] = nums[i - 1], nums[i]
print(nums, is_wiggle(nums))

[3, 5, 1, 6, 2, 4] True


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

**Ý tưởng.** Với `n` nhỏ, duyệt mọi hoán vị của `nums` và lấy hoán vị đầu tiên thỏa `is_wiggle`; ghi vào `nums` tại chỗ. Đúng chắc chắn (theo định nghĩa) nhưng `n!`. Dùng làm chuẩn đối chiếu cho `n ≤ 8`.

In [7]:
def wiggle_bruteforce(nums: list[int]) -> None:
    for p in permutations(nums):
        if is_wiggle(p):
            nums[:] = p                              # ghi đè tại chỗ
            return


for nums, expected in examples:
    a = list(nums)
    wiggle_bruteforce(a)
    print(f"nums={nums} -> {a} hợp lệ? {is_wiggle(a)} và là hoán vị? {sorted(a) == sorted(nums)}")

nums=[3, 5, 2, 1, 6, 4] -> [3, 5, 2, 6, 1, 4] hợp lệ? True và là hoán vị? True
nums=[6, 6, 5, 6, 3, 8] -> [6, 6, 5, 6, 3, 8] hợp lệ? True và là hoán vị? True


**Độ phức tạp (Cách 1).** **Time `O(n · n!)`** (sinh hoán vị và kiểm tra), **Space `O(n)`**. Với `n = 5·10⁴` hoàn toàn không khả thi; chỉ dùng cho `n ≤ 8` làm oracle.

**Nút thắt.** Ta không cần hoán vị **bất kỳ**: chỉ cần thỏa điều kiện **cục bộ** giữa các phần tử kề nhau.

## 6. Ý tưởng thuật toán / bước đột phá

**Điều kiện chỉ ràng buộc các cặp kề nhau**, nên có thể sửa **cục bộ**. Hai hướng:

- **Sắp xếp một lần rồi bố trí:** mảng đã sắp tăng `a`; đổi chỗ từng cặp `(a[1], a[2])`, `(a[3], a[4])`, ... hoặc xen kẽ nửa nhỏ và nửa lớn. Dễ chứng minh nhưng `O(n log n)`.
- **Quét một lượt và hoán đổi nếu vi phạm (tham lam):** với `i = 1..n − 1`: nếu `i` lẻ và `nums[i] < nums[i − 1]`, hoặc `i` chẵn và `nums[i] > nums[i − 1]` thì **đổi chỗ** `nums[i]` và `nums[i − 1]`. `O(n)` thời gian, `O(1)` bộ nhớ phụ.

**Vì sao đổi cục bộ không hỏng phần trước?** Xét `i` lẻ, vi phạm: `nums[i] < nums[i − 1]`. Vị trí `i − 1` là chẵn (đáy), và theo bất biến `nums[i − 2] ≥ nums[i − 1]`. Sau khi đổi, `nums[i − 1]` mới bằng giá trị **nhỏ hơn** cũ, nên `nums[i − 2] ≥` giá trị cũ `>` giá trị mới: cặp `(i − 2, i − 1)` vẫn đúng; cặp `(i − 1, i)` thành `(nhỏ, lớn)`: đúng. Trường hợp `i` chẵn đối xứng.

In [8]:
def check_swap_safe(trials=20000):
    """Kiểm chứng bằng thực nghiệm: sau mỗi bước, nums[0..i] luôn xen kẽ đúng."""
    random.seed(0)
    for _ in range(trials):
        n = random.randint(2, 12)
        a = [random.randint(0, 6) for _ in range(n)]
        for i in range(1, n):
            if (i % 2 == 1 and a[i] < a[i - 1]) or (i % 2 == 0 and a[i] > a[i - 1]):
                a[i], a[i - 1] = a[i - 1], a[i]
            assert is_wiggle(a[:i + 1]), (a, i)
    return True


print("bất biến đúng trên 20000 mảng ngẫu nhiên:", check_swap_safe())

bất biến đúng trên 20000 mảng ngẫu nhiên: True


## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp rồi đổi chỗ từng cặp liền kề

**Giải thích.** Sắp `nums` tăng dần: `a[0] ≤ a[1] ≤ a[2] ≤ ...`. Đổi chỗ `a[1]` với `a[2]`, `a[3]` với `a[4]`, ... (các cặp `(i, i + 1)` với `i` lẻ). Ví dụ `[1,2,3,4,5,6] → [1,3,2,5,4,6]`. Chứng minh ở mục 10.

In [9]:
def wiggle_sort_swap_pairs(nums: list[int]) -> None:
    nums.sort()                                      # tăng dần
    for i in range(1, len(nums) - 1, 2):             # i = 1, 3, 5, ...: đổi (i, i + 1)
        nums[i], nums[i + 1] = nums[i + 1], nums[i]


for nums, expected in examples:
    a = list(nums)
    wiggle_sort_swap_pairs(a)
    print(f"nums={nums} -> {a} hợp lệ? {is_wiggle(a)}")

nums=[3, 5, 2, 1, 6, 4] -> [1, 3, 2, 5, 4, 6] hợp lệ? True
nums=[6, 6, 5, 6, 3, 8] -> [3, 6, 5, 6, 6, 8] hợp lệ? True


**Độ phức tạp Cách 2:** **Time `O(n log n)`** (sắp xếp), **Space `O(log n)` hoặc `O(1)`** (sắp tại chỗ).

### Cách 3 — Sắp xếp rồi xen kẽ nửa nhỏ và nửa lớn

**Giải thích.** Sắp `s = sorted(nums)`. Chia thành nửa **nhỏ** `s[:m]` với `m = (n + 1) // 2` (số vị trí chẵn) và nửa **lớn** `s[m:]`. Đặt nửa nhỏ vào các vị trí chẵn (`nums[0::2]`) và nửa lớn vào các vị trí lẻ (`nums[1::2]`) theo thứ tự. Mỗi đáy (vị trí chẵn) thuộc nửa nhỏ nên `≤` mọi đỉnh (nửa lớn): thỏa mọi so sánh cục bộ.

In [10]:
def wiggle_sort_interleave(nums: list[int]) -> None:
    s = sorted(nums)
    m = (len(nums) + 1) // 2                         # số vị trí chẵn
    nums[0::2] = s[:m]                               # nửa nhỏ vào vị trí chẵn (đáy)
    nums[1::2] = s[m:]                               # nửa lớn vào vị trí lẻ (đỉnh)


for nums, expected in examples:
    a = list(nums)
    wiggle_sort_interleave(a)
    print(f"nums={nums} -> {a} hợp lệ? {is_wiggle(a)}")

nums=[3, 5, 2, 1, 6, 4] -> [1, 4, 2, 5, 3, 6] hợp lệ? True
nums=[6, 6, 5, 6, 3, 8] -> [3, 6, 5, 6, 6, 8] hợp lệ? True


**Độ phức tạp Cách 3:** **Time `O(n log n)`**, **Space `O(n)`** cho `s`. Chứng minh rất gọn vì mọi phần tử vị trí chẵn `≤` mọi phần tử vị trí lẻ.

### Cách 4 — Một lượt, sửa cục bộ bằng hoán đổi (cách của lời giải tham khảo, `O(n)`)

**Giải thích.** Quét `i = 1..n − 1`: nếu `i` lẻ mà `nums[i] < nums[i − 1]`, hoặc `i` chẵn mà `nums[i] > nums[i − 1]` thì đổi chỗ `nums[i]` và `nums[i − 1]`.

In [11]:
def wiggle_sort_one_pass(nums: list[int]) -> None:
    for i in range(1, len(nums)):
        if (i % 2 == 1 and nums[i] < nums[i - 1]) or (i % 2 == 0 and nums[i] > nums[i - 1]):
            nums[i], nums[i - 1] = nums[i - 1], nums[i]     # sửa cặp vi phạm bằng hoán đổi


for nums, expected in examples:
    a = list(nums)
    wiggle_sort_one_pass(a)
    print(f"nums={nums} -> {a} (ví dụ đề {expected}) hợp lệ? {is_wiggle(a)}")

nums=[3, 5, 2, 1, 6, 4] -> [3, 5, 1, 6, 2, 4] (ví dụ đề [3, 5, 1, 6, 2, 4]) hợp lệ? True
nums=[6, 6, 5, 6, 3, 8] -> [6, 6, 5, 6, 3, 8] (ví dụ đề [6, 6, 5, 6, 3, 8]) hợp lệ? True


In [12]:
# Code nộp LeetCode cho Cách 4
from typing import List


class Solution:
    def wiggleSort(self, nums: List[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
        for i in range(1, len(nums)):
            if (i % 2 == 1 and nums[i] < nums[i - 1]) or (
                i % 2 == 0 and nums[i] > nums[i - 1]
            ):
                nums[i], nums[i - 1] = nums[i - 1], nums[i]

**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** (chỉ sửa tại chỗ). Đáp ứng follow up.

### Cách 5 — Một lượt, nhìn về phía trước bằng `min`/`max`

**Giải thích.** Cách viết khác của cùng ý tưởng: với mỗi `i` từ `0` đến `n − 2`, đặt cặp `(nums[i], nums[i + 1])` về dạng "nhỏ rồi lớn" nếu `i` chẵn, "lớn rồi nhỏ" nếu `i` lẻ: `lo, hi = sorted(...)`. Đây là cách nhìn "đỉnh luôn là max của cặp, đáy luôn là min của cặp", **không** cần hai nhánh `if`.

In [13]:
def wiggle_sort_look_ahead(nums: list[int]) -> None:
    for i in range(len(nums) - 1):
        a, b = nums[i], nums[i + 1]
        lo, hi = (a, b) if a <= b else (b, a)
        if i % 2 == 0:
            nums[i], nums[i + 1] = lo, hi            # i chẵn là đáy: nums[i] <= nums[i + 1]
        else:
            nums[i], nums[i + 1] = hi, lo            # i lẻ là đỉnh: nums[i] >= nums[i + 1]


for nums, expected in examples:
    a = list(nums)
    wiggle_sort_look_ahead(a)
    print(f"nums={nums} -> {a} hợp lệ? {is_wiggle(a)}")

nums=[3, 5, 2, 1, 6, 4] -> [3, 5, 1, 6, 2, 4] hợp lệ? True
nums=[6, 6, 5, 6, 3, 8] -> [6, 6, 5, 6, 3, 8] hợp lệ? True


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**. Chứng minh: sau bước `i`, cặp `(i, i + 1)` đúng và các cặp trước vẫn đúng vì bước `i` chỉ có thể **giảm** đáy `nums[i]` (khi `i` chẵn... ) hoặc **tăng** đỉnh; chi tiết ở mục 10.

> Cách 4 là cách nộp bài. Cách 5 là cách nhìn khác cùng độ phức tạp. Cách 2 và 3 dễ chứng minh. Cách 1 là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 4 trên `[3, 5, 2, 1, 6, 4]` và `[4, 3, 2, 1]`, rồi Cách 3 trên `[3, 5, 2, 1, 6, 4]`.

In [14]:
def trace_one_pass(nums):
    a = list(nums)
    print("ban đầu:", a)
    for i in range(1, len(a)):
        kind = "đỉnh (cần >= trước)" if i % 2 == 1 else "đáy (cần <= trước)"
        bad = (i % 2 == 1 and a[i] < a[i - 1]) or (i % 2 == 0 and a[i] > a[i - 1])
        if bad:
            a[i], a[i - 1] = a[i - 1], a[i]
        print(f"  i={i} {kind}: {'VI PHẠM, đổi chỗ' if bad else 'đúng':16} -> {a}")
    print("  => kết quả", a, "hợp lệ?", is_wiggle(a))
    return a


assert is_wiggle(trace_one_pass([3, 5, 2, 1, 6, 4]))
print()
assert is_wiggle(trace_one_pass([4, 3, 2, 1]))

ban đầu: [3, 5, 2, 1, 6, 4]
  i=1 đỉnh (cần >= trước): đúng             -> [3, 5, 2, 1, 6, 4]
  i=2 đáy (cần <= trước): đúng             -> [3, 5, 2, 1, 6, 4]
  i=3 đỉnh (cần >= trước): VI PHẠM, đổi chỗ -> [3, 5, 1, 2, 6, 4]
  i=4 đáy (cần <= trước): VI PHẠM, đổi chỗ -> [3, 5, 1, 6, 2, 4]
  i=5 đỉnh (cần >= trước): đúng             -> [3, 5, 1, 6, 2, 4]
  => kết quả [3, 5, 1, 6, 2, 4] hợp lệ? True

ban đầu: [4, 3, 2, 1]
  i=1 đỉnh (cần >= trước): VI PHẠM, đổi chỗ -> [3, 4, 2, 1]
  i=2 đáy (cần <= trước): đúng             -> [3, 4, 2, 1]
  i=3 đỉnh (cần >= trước): VI PHẠM, đổi chỗ -> [3, 4, 1, 2]
  => kết quả [3, 4, 1, 2] hợp lệ? True


In [15]:
def trace_interleave(nums):
    s = sorted(nums)
    m = (len(nums) + 1) // 2
    print(f"nums={nums}, đã sắp = {s}, nửa nhỏ (vị trí chẵn) = {s[:m]}, nửa lớn (vị trí lẻ) = {s[m:]}")
    out = [None] * len(nums)
    out[0::2] = s[:m]
    out[1::2] = s[m:]
    print("  kết quả:", out, "hợp lệ?", is_wiggle(out))
    return out


assert is_wiggle(trace_interleave([3, 5, 2, 1, 6, 4]))

nums=[3, 5, 2, 1, 6, 4], đã sắp = [1, 2, 3, 4, 5, 6], nửa nhỏ (vị trí chẵn) = [1, 2, 3], nửa lớn (vị trí lẻ) = [4, 5, 6]
  kết quả: [1, 4, 2, 5, 3, 6] hợp lệ? True


Bảng tóm tắt (Cách 4, `[3, 5, 2, 1, 6, 4]`):

| `i` | vai trò | cặp trước khi sửa | hành động | mảng sau bước |
|---|---|---|---|---|
| 1 | đỉnh | `(3, 5)` | đúng | `[3, 5, 2, 1, 6, 4]` |
| 2 | đáy | `(5, 2)` | đúng | `[3, 5, 2, 1, 6, 4]` |
| 3 | đỉnh | `(2, 1)` | đổi chỗ | `[3, 5, 1, 2, 6, 4]` |
| 4 | đáy | `(2, 6)` | đổi chỗ | `[3, 5, 1, 6, 2, 4]` |
| 5 | đỉnh | `(2, 4)` | đúng | `[3, 5, 1, 6, 2, 4]` |

Chú ý bước `i = 4`: đổi `(2, 6)` thành `(6, 2)`, làm đỉnh `6` đứng cạnh `1` ở bên trái: `1 ≤ 6` vẫn đúng nhờ `6` **lớn hơn** `2`, và `2 ≥ 1`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4. Đúng class `Solution`, đúng method `wiggleSort(self, nums: List[int]) -> None`: **sửa `nums` tại chỗ, không trả về gì**.

In [16]:
from typing import List


class Solution:
    def wiggleSort(self, nums: List[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
        for i in range(1, len(nums)):
            if (i % 2 == 1 and nums[i] < nums[i - 1]) or (    # vị trí lẻ (đỉnh) nhỏ hơn phần tử trước
                i % 2 == 0 and nums[i] > nums[i - 1]          # vị trí chẵn (đáy) lớn hơn phần tử trước
            ):
                nums[i], nums[i - 1] = nums[i - 1], nums[i]   # đổi chỗ để sửa cặp vi phạm


for nums, expected in examples:
    a = list(nums)
    result = Solution().wiggleSort(a)
    assert result is None and is_wiggle(a) and sorted(a) == sorted(nums)
    assert a == expected                                       # với Cách 4 chính xác bằng ví dụ của đề
print("PASS ví dụ đề cho class Solution (sửa tại chỗ, trả None, trùng ví dụ đề)")

PASS ví dụ đề cho class Solution (sửa tại chỗ, trả None, trùng ví dụ đề)


## 10. Vì sao thuật toán đúng

**Bất biến (Cách 4):** sau khi xử lý bước `i`, mảng con `nums[0..i]` thỏa điều kiện xen kẽ (`≤ / ≥` đúng cho mọi cặp kề trong đoạn đó). Cơ sở: `i = 0` (một phần tử) hiển nhiên. Bước quy nạp xét `i ≥ 1`, giả sử `nums[0..i−1]` đúng:

- **`i` lẻ (đỉnh):** cần `nums[i − 1] ≤ nums[i]`. Nếu đã đúng thì không làm gì. Nếu `nums[i] < nums[i − 1]` ta đổi chỗ: cặp `(i − 1, i)` thành `(nhỏ, lớn)` đúng. Cặp `(i − 2, i − 1)` (nếu có; `i − 2` lẻ, `i − 1` chẵn) cần `nums[i − 2] ≥ nums[i − 1]`: trước khi đổi đã có `nums[i − 2] ≥ nums[i − 1]` (cũ) và giá trị mới ở vị trí `i − 1` là `nums[i] < nums[i − 1]` (cũ) nên càng nhỏ: vẫn `nums[i − 2] ≥` mới. Vậy đúng.
- **`i` chẵn (đáy):** cần `nums[i − 1] ≥ nums[i]`. Nếu `nums[i] > nums[i − 1]` ta đổi: cặp `(i − 1, i)` thành `(lớn, nhỏ)` đúng; cặp `(i − 2, i − 1)` (`i − 2` chẵn, `i − 1` lẻ) cần `nums[i − 2] ≤ nums[i − 1]`: trước đó `nums[i − 2] ≤ nums[i − 1]` (cũ) và giá trị mới ở vị trí `i − 1` là `nums[i] > nums[i − 1]` (cũ), càng lớn: vẫn đúng.

Kết thúc ở `i = n − 1`: toàn mảng xen kẽ đúng. Phép đổi chỗ chỉ hoán đổi hai phần tử nên giữ nguyên đa tập (là hoán vị của đầu vào).

**Cách 2 (sắp rồi đổi cặp):** sau sắp, `a[0] ≤ a[1] ≤ ... ≤ a[n − 1]`; gọi `b` là mảng sau khi đổi `(1, 2), (3, 4), ...`. Với mỗi vị trí **lẻ** `i`: nếu `i + 1 < n` thì `b[i] = a[i + 1]`; nếu `i = n − 1` thì `b[i] = a[n − 1]`. Hai hàng xóm `b[i − 1]` và `b[i + 1]` đều là các phần tử gốc có chỉ số **không vượt quá** chỉ số của `b[i]` (hàng xóm trái là `a[i − 1]` hoặc `a[i − 2]`; hàng xóm phải là `a[i]`), nên theo tính tăng của `a` đều `≤ b[i]`. Vậy mọi vị trí lẻ là đỉnh `≥` cả hai hàng xóm, tức mọi cặp kề đều thỏa.

**Cách 3 (xen kẽ nửa nhỏ/lớn):** mỗi phần tử ở vị trí chẵn thuộc nửa nhỏ nên `≤` mọi phần tử nửa lớn; vị trí lẻ thuộc nửa lớn. Mọi cặp kề gồm một chẵn (đáy) và một lẻ (đỉnh) nên đáy `≤` đỉnh: đúng.

**Cách 5:** ở bước `i`, ta thay `(nums[i], nums[i + 1])` bằng `(min, max)` (nếu `i` chẵn) hoặc `(max, min)` (nếu `i` lẻ). Với `i` chẵn, vị trí `i` (đáy) chỉ có thể **giảm** (`min ≤ nums[i]`), nên cặp trước `(nums[i − 1], nums[i])` với `nums[i − 1] ≥ nums[i]` (cũ) vẫn đúng; `i` lẻ đối xứng (đỉnh chỉ **tăng**).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- **Cách 1 (thử mọi hoán vị):** Time `O(n · n!)`, Space `O(n)`.
- **Cách 2 (sắp xếp + đổi cặp):** Time `O(n log n)`, Space `O(log n)` hoặc `O(1)`.
- **Cách 3 (sắp xếp + xen kẽ):** Time `O(n log n)`, Space `O(n)`.
- **Cách 4 (một lượt, hoán đổi cục bộ):** Time **`O(n)`**, Space **`O(1)`**.
- **Cách 5 (một lượt, min/max):** Time `O(n)`, Space `O(1)`.
- Trường hợp trung bình và xấu nhất như nhau (luôn duyệt hết mảng). Không thể tốt hơn `O(n)` vì phải xem mọi phần tử.

## 12. Bộ test

Vì đáp án **không duy nhất**, ta kiểm tra bằng **tính chất**: (1) kết quả là **hoán vị** của đầu vào (cùng đa tập); (2) thỏa `is_wiggle`; (3) hàm **trả `None`** và sửa tại chỗ. `oracle` độc lập: với mảng nhỏ (`n ≤ 7`) kiểm tra rằng tồn tại hoán vị hợp lệ (đề đảm bảo) bằng cách duyệt mọi hoán vị, và hoán vị mà thuật toán trả về nằm trong tập các hoán vị hợp lệ.

In [17]:
sol = Solution()
impls = {
    "bruteforce": wiggle_bruteforce,
    "swap_pairs": wiggle_sort_swap_pairs,
    "interleave": wiggle_sort_interleave,
    "one_pass": wiggle_sort_one_pass,
    "look_ahead": wiggle_sort_look_ahead,
    "submit": sol.wiggleSort,
}
SMALL_ONLY = ("bruteforce",)


def oracle_valid_set(nums):
    """Tập mọi hoán vị hợp lệ (đa tập các tuple) của nums nhỏ."""
    return {p for p in permutations(nums) if is_wiggle(p)}


def check_result(original, result):
    return sorted(result) == sorted(original) and is_wiggle(result)


def run_all(nums, names=None):
    out = {}
    for name, fn in impls.items():
        if names is not None and name not in names:
            continue
        a = list(nums)
        ret = fn(a)
        assert ret is None, (name, "phải sửa tại chỗ và trả None")
        out[name] = a
    return out


for nums, expected in examples:
    valid = oracle_valid_set(nums)
    assert tuple(expected) in valid
    for name, a in run_all(nums).items():
        assert check_result(nums, a), (name, nums, a)
        assert tuple(a) in valid, (name, nums, a)
print("PASS ví dụ của đề (mọi cách trả hoán vị hợp lệ nằm trong tập oracle)")

PASS ví dụ của đề (mọi cách trả hoán vị hợp lệ nằm trong tập oracle)


### 12.1. Các trường hợp biên

In [18]:
def check_all(nums, label):
    names = None if len(nums) <= 8 else [n for n in impls if n not in SMALL_ONLY]
    for name, a in run_all(nums, names).items():
        assert check_result(nums, a), (label, name, nums, a)
    print("PASS", label)


check_all([1], "n = 1")
check_all([2, 1], "n = 2 giảm dần -> [1, 2]")
check_all([1, 2], "n = 2 tăng dần")
check_all([5, 5], "n = 2 bằng nhau")
check_all([1, 2, 3], "n = 3 tăng dần")
check_all([3, 2, 1], "n = 3 giảm dần")
check_all([2, 2, 2, 2], "toàn giá trị bằng nhau")
check_all([0, 0, 1, 1], "hai giá trị lặp")
check_all(list(range(10)), "mảng tăng dần n = 10")
check_all(list(range(10, 0, -1)), "mảng giảm dần n = 10")
check_all([6, 6, 5, 6, 3, 8], "ví dụ 2 của đề")
check_all([10000, 0] * 10, "giá trị biên 0 và 10^4")
check_all(list(range(50000)), "n = 5*10^4 tăng dần")
check_all(list(range(50000, 0, -1)), "n = 5*10^4 giảm dần")
check_all([random.randint(0, 10000) for _ in range(50000)], "n = 5*10^4 ngẫu nhiên")

PASS n = 1
PASS n = 2 giảm dần -> [1, 2]
PASS n = 2 tăng dần
PASS n = 2 bằng nhau
PASS n = 3 tăng dần
PASS n = 3 giảm dần
PASS toàn giá trị bằng nhau
PASS hai giá trị lặp
PASS mảng tăng dần n = 10
PASS mảng giảm dần n = 10
PASS ví dụ 2 của đề
PASS giá trị biên 0 và 10^4
PASS n = 5*10^4 tăng dần
PASS n = 5*10^4 giảm dần


PASS n = 5*10^4 ngẫu nhiên


### 12.2. Tính chất và phản ví dụ

In [19]:
random.seed(280)

# Tính chất 1: mảng đã đúng thì Cách 4 không đổi gì
for _ in range(300):
    n = random.randint(1, 15)
    a = sorted(random.randint(0, 20) for _ in range(n))
    Solution().wiggleSort(a)
    b = a[:]
    Solution().wiggleSort(b)
    assert a == b                                   # lặp lại lần hai không đổi (đã thỏa)
print("PASS tính chất: gọi hai lần không thay đổi thêm (đã thỏa sau lần đầu)")

# Tính chất 2: số lần hoán đổi của Cách 4 <= n - 1
def count_swaps(nums):
    a = list(nums)
    swaps = 0
    for i in range(1, len(a)):
        if (i % 2 == 1 and a[i] < a[i - 1]) or (i % 2 == 0 and a[i] > a[i - 1]):
            a[i], a[i - 1] = a[i - 1], a[i]
            swaps += 1
    return swaps

for _ in range(300):
    n = random.randint(1, 30)
    a = [random.randint(0, 9) for _ in range(n)]
    assert count_swaps(a) <= max(0, n - 1)
print("PASS tính chất: Cách 4 thực hiện tối đa n - 1 hoán đổi")

# Tính chất 3: các cách cho kết quả khác nhau nhưng đều hợp lệ (đáp án không duy nhất)
different = 0
for _ in range(100):
    a = [random.randint(0, 9) for _ in range(random.randint(4, 8))]
    results = {tuple(v) for v in run_all(a, ["swap_pairs", "interleave", "one_pass", "look_ahead"]).values()}
    different += len(results) > 1
print("số mảng mà các cách cho kết quả khác nhau (đều hợp lệ):", different)

# Phản ví dụ: chỉ sort() không đủ
def sai_chi_sort(nums):
    nums.sort()

a = [3, 5, 2, 1, 6, 4]
sai_chi_sort(a)
print("chỉ sort ->", a, "hợp lệ?", is_wiggle(a))
assert not is_wiggle(a)

PASS tính chất: gọi hai lần không thay đổi thêm (đã thỏa sau lần đầu)
PASS tính chất: Cách 4 thực hiện tối đa n - 1 hoán đổi
số mảng mà các cách cho kết quả khác nhau (đều hợp lệ): 94
chỉ sort -> [1, 2, 3, 4, 5, 6] hợp lệ? False


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [20]:
random.seed(2026)
for trial in range(1500):                           # n <= 7: so với tập hoán vị hợp lệ (oracle brute force)
    n = random.randint(1, 7)
    nums = [random.randint(0, random.choice([2, 5, 10])) for _ in range(n)]
    valid = oracle_valid_set(nums)
    assert valid, "đề đảm bảo có đáp án"
    for name, a in run_all(nums).items():
        assert tuple(a) in valid, (name, nums, a)

for trial in range(40):                             # mảng dài: kiểm tra tính chất
    n = random.randint(1000, 50000)
    nums = [random.randint(0, random.choice([3, 100, 10000])) for _ in range(n)]
    for name, a in run_all(nums, [x for x in impls if x not in SMALL_ONLY]).items():
        assert check_result(nums, a), (name, n)
print("PASS 1500 mảng nhỏ nằm trong tập hoán vị hợp lệ của oracle; 40 mảng dài thỏa tính chất")

PASS 1500 mảng nhỏ nằm trong tập hoán vị hợp lệ của oracle; 40 mảng dài thỏa tính chất


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Xét sai chẵn/lẻ (đảo chiều so sánh)

In [21]:
def sai_dao_chieu(nums):
    for i in range(1, len(nums)):
        if (i % 2 == 0 and nums[i] < nums[i - 1]) or (i % 2 == 1 and nums[i] > nums[i - 1]):     # đảo vai đỉnh/đáy
            nums[i], nums[i - 1] = nums[i - 1], nums[i]


a = [3, 5, 2, 1, 6, 4]
sai_dao_chieu(a)
print("sai:", a, "hợp lệ?", is_wiggle(a), "(cho ra dạng >= <= >= ... ngược yêu cầu)")

sai: [5, 2, 3, 1, 6, 4] hợp lệ? False (cho ra dạng >= <= >= ... ngược yêu cầu)


**Vì sao sai:** yêu cầu là `nums[0] ≤ nums[1] ≥ ...` nên **chỉ số lẻ là đỉnh** (`≥ phần tử trước`) và **chỉ số chẵn là đáy** (`≤ phần tử trước`). Đảo hai điều kiện cho dãy `≥ ≤ ≥ ...` (xuất phát bằng "xuống"), không thỏa đề (trừ khi mảng có nhiều giá trị bằng nhau).

### Lỗi 2: Chỉ `sort()` rồi dừng

In [22]:
a = [3, 5, 2, 1, 6, 4]
a.sort()
print("sort():", a, "hợp lệ?", is_wiggle(a))
b = [1, 2, 3, 4, 5, 6]
Solution().wiggleSort(b)
print("sau khi đổi cặp / sửa cục bộ:", b, "hợp lệ?", is_wiggle(b))

sort(): [1, 2, 3, 4, 5, 6] hợp lệ? False
sau khi đổi cặp / sửa cục bộ: [1, 3, 2, 5, 4, 6] hợp lệ? True


**Vì sao sai:** mảng đã sắp tăng `1 ≤ 2 ≤ 3 ≤ ...` thỏa `≤` ở mọi cặp nhưng **vi phạm `≥`** ở các vị trí lẻ (`2 ≥ 3` sai). Cần thêm bước đổi cặp hoặc xen kẽ.

### Lỗi 3: Dùng `nums[i + 1]` mà không kiểm tra biên

In [23]:
def sai_vuot_bien(nums):
    for i in range(len(nums)):                          # sai: i + 1 vượt biên ở phần tử cuối
        if i % 2 == 0 and nums[i] > nums[i + 1]:
            nums[i], nums[i + 1] = nums[i + 1], nums[i]
        elif i % 2 == 1 and nums[i] < nums[i + 1]:
            nums[i], nums[i + 1] = nums[i + 1], nums[i]


try:
    sai_vuot_bien([3, 5, 2, 1, 6, 4])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** ở phần tử cuối `i = n − 1`, `nums[i + 1]` không tồn tại. Nếu nhìn về phía trước phải lặp `i` đến `n − 2` (`range(n − 1)`); nếu nhìn về phía sau thì lặp `i` từ `1` (`nums[i − 1]`).

### Lỗi 4: Trả về mảng mới thay vì sửa tại chỗ

In [24]:
class SaiTraVe:
    def wiggleSort(self, nums):
        result = sorted(nums)                           # tạo list mới
        for i in range(1, len(result) - 1, 2):
            result[i], result[i + 1] = result[i + 1], result[i]
        return result                                   # sai: người chấm kiểm tra nums, không dùng giá trị trả về


nums = [3, 5, 2, 1, 6, 4]
ret = SaiTraVe().wiggleSort(nums)
print("giá trị trả về:", ret, "hợp lệ?", is_wiggle(ret))
print("nums sau khi gọi:", nums, "hợp lệ?", is_wiggle(nums))

giá trị trả về: [1, 3, 2, 5, 4, 6] hợp lệ? True
nums sau khi gọi: [3, 5, 2, 1, 6, 4] hợp lệ? False


**Vì sao sai:** đề yêu cầu "**sửa `nums` tại chỗ**, không trả về gì". Hàm trên tạo list mới nên `nums` gốc **không đổi** và bị chấm sai. Dùng `nums[:] = ...`, `nums.sort()`, hoặc hoán đổi trực tiếp các phần tử của `nums`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Điều kiện chỉ ràng buộc các cặp **kề nhau** (zigzag/wiggle): **không cần sắp xếp toàn bộ**; sửa **cục bộ** bằng hoán đổi trong một lượt `O(n)`.
- Khuôn "quét và sửa cặp vi phạm": kiểm tra vị trí lẻ/chẵn, đổi chỗ với phần tử trước; chứng minh bằng **bất biến** (phần đã xử lý luôn đúng).
- Nếu dễ chứng minh hơn là nhanh: **sắp xếp rồi bố trí** (đổi cặp hoặc xen kẽ nửa nhỏ/nửa lớn).
- Bài nâng cao: **Wiggle Sort II (LeetCode 324)** yêu cầu **thật chặt** `<` và `>`, cần chia đôi và xen kẽ cẩn thận (không dùng được cách đơn giản này khi có phần tử lặp).
- Khi hàm yêu cầu "**in-place**", dùng hoán đổi / gán lát cắt `nums[:] = ...`, không `return` mảng mới.
- Test bài có nhiều đáp án: kiểm tra **tính chất** (hoán vị + điều kiện), đối chiếu oracle brute force trên mảng nhỏ.

## 15. Biến thể của bài

1. **Wiggle Sort II (LeetCode 324):** `nums[0] < nums[1] > nums[2] < ...` (chặt).
2. **Dạng ngược:** `nums[0] >= nums[1] <= nums[2] >= ...`.
3. **Đếm số hoán đổi tối thiểu** hoặc số cặp vi phạm ban đầu.
4. **Kiểm tra mảng có phải wiggle** (như `is_wiggle`).
5. **Wiggle Subsequence (LeetCode 376):** độ dài dãy con "lắc" dài nhất.

In [25]:
def wiggle_sort_reverse(nums):
    """Biến thể 2: nums[0] >= nums[1] <= nums[2] >= ..."""
    for i in range(1, len(nums)):
        if (i % 2 == 1 and nums[i] > nums[i - 1]) or (i % 2 == 0 and nums[i] < nums[i - 1]):
            nums[i], nums[i - 1] = nums[i - 1], nums[i]


def is_wiggle_strict(a):
    """Biến thể 1 / 4: a[0] < a[1] > a[2] < ..."""
    return all((a[i] < a[i + 1]) if i % 2 == 0 else (a[i] > a[i + 1]) for i in range(len(a) - 1))


def wiggle_sort_strict(nums):
    """Biến thể 1 (cách đơn giản, đúng khi có đáp án): sắp rồi xen nửa nhỏ (đảo ngược) và nửa lớn (đảo ngược)."""
    s = sorted(nums)
    m = (len(nums) + 1) // 2
    nums[0::2] = s[:m][::-1]                  # nửa nhỏ đảo ngược
    nums[1::2] = s[m:][::-1]                  # nửa lớn đảo ngược


def count_violations(nums):
    """Biến thể 3: số cặp kề vi phạm."""
    return sum(not ((nums[i] <= nums[i + 1]) if i % 2 == 0 else (nums[i] >= nums[i + 1])) for i in range(len(nums) - 1))


def wiggle_subsequence_length(nums):
    """Biến thể 5 (LeetCode 376): độ dài dãy con lắc dài nhất, O(n)."""
    if len(nums) < 2:
        return len(nums)
    up = down = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            up = down + 1
        elif nums[i] < nums[i - 1]:
            down = up + 1
    return max(up, down)


a = [3, 5, 2, 1, 6, 4]
wiggle_sort_reverse(a)
assert all((a[i] >= a[i + 1]) if i % 2 == 0 else (a[i] <= a[i + 1]) for i in range(len(a) - 1))
b = [1, 2, 3, 4, 5, 6]
wiggle_sort_strict(b)
assert is_wiggle_strict(b), b
assert count_violations([1, 2, 3, 4, 5]) == 2 and count_violations([3, 5, 1, 6, 2, 4]) == 0
assert wiggle_subsequence_length([1, 7, 4, 9, 2, 5]) == 6 and wiggle_subsequence_length([1, 17, 5, 10, 13, 15, 10, 5, 16, 8]) == 7
print("PASS biến thể: dạng ngược, chặt, đếm vi phạm, dãy con lắc")

PASS biến thể: dạng ngược, chặt, đếm vi phạm, dãy con lắc


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi hoán vị | `permutations` + `is_wiggle` | `O(n·n!)` | `O(n)` | oracle `n ≤ 8` |
| 2. Sắp xếp + đổi cặp | đổi `(1,2), (3,4), ...` | `O(n log n)` | `O(1)`–`O(log n)` | dễ chứng minh |
| 3. Sắp xếp + xen kẽ nửa nhỏ/lớn | nửa nhỏ vào vị trí chẵn | `O(n log n)` | `O(n)` | chứng minh rất gọn |
| 4. Một lượt hoán đổi cục bộ | sửa cặp `(i − 1, i)` vi phạm | **`O(n)`** | `O(1)` | **cách nộp bài** (follow up) |
| 5. Một lượt min/max | cặp `(i, i + 1)` thành (min, max)/(max, min) | `O(n)` | `O(1)` | cách viết khác, không rẽ nhánh `if` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `is_wiggle([1, 3, 2])`, `is_wiggle([1, 2, 3])`, `[10, 11, 12, 13][1::2]`, danh sách `a` sau `a = [2, 1]; Solution().wiggleSort(a)`, và `Solution().wiggleSort([5])` trả về gì?

**Bài 2 (tính tay).** Chạy Cách 4 bằng tay trên `[4, 3, 2, 1]` và `[1, 2, 3, 4, 5]`, ghi lại mảng sau từng bước `i`.

**Bài 3 (điền TODO).** Hoàn thành `wiggle_todo(nums)` theo Cách 4.

In [26]:
def wiggle_todo(nums):
    for i in range(1, len(nums)):
        # TODO 1: nếu i lẻ và nums[i] < nums[i - 1], hoặc i chẵn và nums[i] > nums[i - 1]
        #     TODO 2: đổi chỗ nums[i] và nums[i - 1]
        pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `wiggle_interleave2(nums)` theo Cách 3 (sắp xếp rồi đặt nửa nhỏ vào chỉ số chẵn, nửa lớn vào chỉ số lẻ). Kiểm tra bằng hai ví dụ của đề bằng `is_wiggle`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `is_wiggle_reverse(nums)` kiểm tra dạng `nums[0] >= nums[1] <= nums[2] >= ...`. (b) Viết `min_swaps_to_wiggle(nums)` trả về số lần hoán đổi **kề nhau** mà Cách 4 thực hiện (cũng là một cận trên cho số hoán đổi tối thiểu).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: vị trí chẵn là đáy (`≤` phần tử sau), vị trí lẻ là đỉnh (`≥` phần tử sau); lát cắt `[1::2]` lấy chỉ số lẻ; hàm trả `None`. Bài 2: kiểm tra cặp `(nums[i − 1], nums[i])` theo `i` chẵn/lẻ.
- **Mức 2 (vừa):** Bài 1: `True`, `False`, `[11, 13]`, `[1, 2]`, `None`. Bài 2: `[4,3,2,1]`: `i=1` đỉnh `3 < 4` đổi `[3,4,2,1]`; `i=2` đáy `2 ≤ 4` đúng; `i=3` đỉnh `1 < 2` đổi `[3,4,1,2]`. `[1,2,3,4,5]`: `i=1` đúng; `i=2` đáy `3 > 2` đổi `[1,3,2,4,5]`; `i=3` đỉnh `4 ≥ 2` đúng; `i=4` đáy `5 > 4` đổi `[1,3,2,5,4]`. Bài 3: hai TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `s = sorted(nums); m = (len(nums) + 1) // 2; nums[0::2] = s[:m]; nums[1::2] = s[m:]`. Bài 5(a): đảo hai vai chẵn/lẻ so với `is_wiggle`. Bài 5(b): đếm số lần đổi chỗ trong vòng lặp của Cách 4 (đã có `count_swaps`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert is_wiggle([1, 3, 2]) is True and is_wiggle([1, 2, 3]) is False
assert [10, 11, 12, 13][1::2] == [11, 13]
a = [2, 1]
assert Solution().wiggleSort(a) is None and a == [1, 2]
assert Solution().wiggleSort([5]) is None

# Bài 2
a = [4, 3, 2, 1]
Solution().wiggleSort(a)
assert a == [3, 4, 1, 2]
a = [1, 2, 3, 4, 5]
Solution().wiggleSort(a)
assert a == [1, 3, 2, 5, 4]

# Bài 3
def wiggle_todo_ans(nums):
    for i in range(1, len(nums)):
        if (i % 2 == 1 and nums[i] < nums[i - 1]) or (i % 2 == 0 and nums[i] > nums[i - 1]):
            nums[i], nums[i - 1] = nums[i - 1], nums[i]

for nums, expected in examples:
    a = list(nums)
    wiggle_todo_ans(a)
    assert a == expected

# Bài 4
def wiggle_interleave2(nums):
    s = sorted(nums)
    m = (len(nums) + 1) // 2
    nums[0::2] = s[:m]
    nums[1::2] = s[m:]

for nums, expected in examples:
    a = list(nums)
    wiggle_interleave2(a)
    assert is_wiggle(a) and sorted(a) == sorted(nums)

# Bài 5(a), 5(b)
def is_wiggle_reverse(nums):
    return all((nums[i] >= nums[i + 1]) if i % 2 == 0 else (nums[i] <= nums[i + 1]) for i in range(len(nums) - 1))

def min_swaps_to_wiggle(nums):
    return count_swaps(nums)

assert is_wiggle_reverse([3, 1, 2]) and not is_wiggle_reverse([1, 2, 3])
assert min_swaps_to_wiggle([3, 5, 2, 1, 6, 4]) == 2
random.seed(19)
for _ in range(200):
    nums = [random.randint(0, 9) for _ in range(random.randint(1, 12))]
    b = list(nums)
    wiggle_sort_reverse(b)
    assert is_wiggle_reverse(b)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (quét một lượt, hoán đổi cặp vi phạm)

```python
class Solution:
    def wiggleSort(self, nums: List[int]) -> None:
        for i in range(1, len(nums)):
            if (i % 2 == 1 and nums[i] < nums[i - 1]) or (i % 2 == 0 and nums[i] > nums[i - 1]):
                nums[i], nums[i - 1] = nums[i - 1], nums[i]
```

**Checklist**

- [ ] Vị trí **lẻ** là đỉnh (`nums[i] ≥ nums[i − 1]`), vị trí **chẵn** là đáy (`nums[i] ≤ nums[i − 1]`)?
- [ ] Chỉ số bắt đầu từ `1` và so với `nums[i − 1]` (không vượt biên)?
- [ ] **Sửa tại chỗ**, hàm trả `None` (không `return` mảng mới)?
- [ ] Không chỉ `sort()` (không đủ)?
- [ ] Nhớ đây là bản **không chặt** (`≤`, `≥`); bản chặt (LeetCode 324) khác?
- [ ] Đã thử: `n = 1`, `n = 2`, mảng tăng dần, giảm dần, toàn giá trị bằng nhau?
- [ ] Đã kiểm tra kết quả là hoán vị của đầu vào và thỏa điều kiện (và đối chiếu brute force trên mảng nhỏ)?
- [ ] Skeleton đúng `class Solution` / `def wiggleSort(self, nums: List[int]) -> None`?